In [5]:
# =============================================================================
# TASK 1: Graph Concepts & State Design
# =============================================================================
# LANGGRAPH CORE BUILDING BLOCKS:
#
# 1. StateGraph — Container for your workflow (like a blank flowchart)
# 2. State — Shared TypedDict that all nodes read from and write to
# 3. Node — A function that takes state, does work, returns updated fields
# 4. Edge — Transition to next node (add_edge "plan" -> "execute")
# 5. Conditional Edge — Smart transition: "if score >= 7 go here, else go there"
# 6. Entry Point — Where execution starts (set_entry_point)
# 7. END — Special node that stops the graph
# =============================================================================

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
import operator

# --- STATE SCHEMA ---
class ResearchState(TypedDict):
    query: str
    plan: str
    search_results: str
    answer: str
    quality_score: int
    retry_count: int
    max_retries: int

# --- ASCII DIAGRAM OF THE GRAPH ---
diagram = """
+-------------------------------------------------------------------+
|                    RESEARCH ASSISTANT GRAPH                        |
+-------------------------------------------------------------------+
|  +----------+                                                     |
|  |  START   |                                                     |
|  +----+-----+                                                     |
|       v                                                            |
|  +----------+    "Create a plan based on the query"               |
|  |   PLAN   +----------------------------------------+            |
|  +----+-----+                                        |            |
|       v                                              |            |
|  +----------+    "Search and retrieve info"          |            |
|  | EXECUTE  |<-------------------------------+      |            |
|  +----+-----+                                |      |            |
|       v                                      |      |            |
|  +----------+    "Write an answer"           |      |            |
|  | GENERATE +--------------------------------+      |            |
|  +----+-----+                                |      |            |
|       v                                      |      |            |
|  +----------+    "Rate quality 0-10"         |      |            |
|  | CRITIQUE +--------------------------------+      |            |
|  +----+-----+                                      |            |
|       |                                            |            |
|       | if score < 7 AND retries < max             |            |
|       +-------------- RETRY LOOP ------------------+            |
|                                                                    |
|       | if score >= 7 OR retries >= max                           |
|       v                                                            |
|  +----------+                                                     |
|  |   END    |                                                     |
|  +----------+                                                     |
+-------------------------------------------------------------------+
"""
print(diagram)

# --- NODES ---

def plan_node(state):
    """Node 1: Create a plan based on the query."""
    query = state["query"]
    plan = f"Search for '{query}', then summarize findings"
    print(f"  [PLAN] {plan}")
    return {"plan": plan}

def execute_node(state):
    """Node 2: Execute the plan (simulate search)."""
    results = f"Found info about: {state['query']}. LangGraph is a graph-based agent framework."
    print(f"  [EXECUTE] Got results")
    return {"search_results": results}

def generate_node(state):
    """Node 3: Generate an answer from search results."""
    answer = f"Based on research: {state['search_results']}. LangGraph gives branching and self-correction."
    print(f"  [GENERATE] Generated answer")
    return {"answer": answer}

def critique_node(state):
    """Node 4: Critique the answer and assign quality score."""
    retry = state["retry_count"]
    score = 5 if retry == 0 else 8
    print(f"  [CRITIQUE] Score: {score}/10 (attempt #{retry + 1})")
    return {
        "quality_score": score,
        "retry_count": retry + 1,
    }

# --- CONDITIONAL ROUTER ---
def route_after_critique(state):
    score = state["quality_score"]
    retries = state["retry_count"]
    max_retries = state["max_retries"]
    if score >= 7:
        print(f"  [ROUTE] Score {score} >= 7 -> FINISH")
        return "finish"
    elif retries < max_retries:
        print(f"  [ROUTE] Score {score} < 7, retries {retries} < {max_retries} -> RETRY")
        return "retry"
    else:
        print(f"  [ROUTE] Max retries reached -> FINISH")
        return "finish"

# --- BUILD THE GRAPH ---
graph = StateGraph(ResearchState)

graph.add_node("plan", plan_node)
graph.add_node("execute", execute_node)
graph.add_node("generate", generate_node)
graph.add_node("critique", critique_node)

graph.add_edge("plan", "execute")
graph.add_edge("execute", "generate")
graph.add_edge("generate", "critique")

graph.add_conditional_edges("critique", route_after_critique, {
    "retry": "execute",
    "finish": END
})

graph.set_entry_point("plan")
app = graph.compile()

# --- RUN IT ---
print("=" * 60)
result = app.invoke({
    "query": "What is LangGraph?",
    "plan": "", "search_results": "", "answer": "",
    "quality_score": 0, "retry_count": 0, "max_retries": 3,
})

print(f"\nFINAL ANSWER: {result['answer'][:80]}...")
print(f"QUALITY SCORE: {result['quality_score']}/10")
print(f"RETRIES: {result['retry_count']}")


+-------------------------------------------------------------------+
|                    RESEARCH ASSISTANT GRAPH                        |
+-------------------------------------------------------------------+
|  +----------+                                                     |
|  |  START   |                                                     |
|  +----+-----+                                                     |
|       v                                                            |
|  +----------+    "Create a plan based on the query"               |
|  |   PLAN   +----------------------------------------+            |
|  +----+-----+                                        |            |
|       v                                              |            |
|  +----------+    "Search and retrieve info"          |            |
|  | EXECUTE  |<-------------------------------+      |            |
|  +----+-----+                                |      |            |
|       v          

In [12]:
# =============================================================================
# TASK 2: Build a Linear Graph
# =============================================================================
# 4 nodes in a straight line: PLAN -> EXECUTE -> GENERATE -> FORMAT
# Uses Day 2 tools + LLM (Groq)
# =============================================================================

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from typing import TypedDict
from langgraph.graph import StateGraph, END

load_dotenv()

# --- LLM ---
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, max_tokens=1024)

# --- TOOLS (from Day 2) ---
@tool
def get_weather(city: str) -> str:
    """Returns current temperature for a given city. Use when user asks about weather."""
    fake_temps = {"tokyo": 22, "paris": 18, "new york": 25, "london": 15}
    temp = fake_temps.get(city.lower(), 20)
    return f"Weather in {city.title()}: {temp}C"

@tool
def product_lookup(product_name: str) -> str:
    """Looks up product info including price and stock. Use when user asks about products."""
    products = {"laptop": 999, "phone": 699, "tablet": 449, "headphones": 149}
    price = products.get(product_name.lower())
    if price:
        return f"Product: {product_name}, Price: ${price}"
    return f"Product '{product_name}' not found"

tools = [get_weather, product_lookup]

# --- STATE SCHEMA ---
class ResearchState(TypedDict):
    query: str
    plan: str
    tool_output: str
    answer: str
    formatted_output: str

# --- NODE 1: PLAN ---
def plan_node(state):
    """Use LLM to create a plan for answering the query."""
    response = llm.invoke([
        {"role": "system", "content": "You are a planner. Create a short step-by-step plan. Reply with just the plan."},
        {"role": "user", "content": state["query"]}
    ])
    plan = response.content
    print(f"  [PLAN] {plan[:120]}...")
    return {"plan": plan}

# --- NODE 2: EXECUTE ---
def execute_node(state):
    """Execute the plan by calling appropriate tools."""
    query = state["query"].lower()

    if "weather" in query or "temperature" in query:
        # Use LLM to extract city from query
        response = llm.invoke([
            {"role": "system", "content": "Extract the city name from the query. Reply with ONLY the city name, nothing else."},
            {"role": "user", "content": state["query"]}
        ])
        city = response.content.strip().lower()
        result = get_weather.invoke({"city": city})
    elif "product" in query or "price" in query:
        # Use LLM to extract product from query
        response = llm.invoke([
            {"role": "system", "content": "Extract the product name from the query. Reply with ONLY the product name, nothing else."},
            {"role": "user", "content": state["query"]}
        ])
        product = response.content.strip().lower()
        result = product_lookup.invoke({"product_name": product})
    else:
        response = llm.invoke([
            {"role": "system", "content": "Answer briefly in 1-2 sentences."},
            {"role": "user", "content": state["query"]}
        ])
        result = response.content

    print(f"  [EXECUTE] {result[:120]}")
    return {"tool_output": result}

# --- NODE 3: GENERATE ---
def generate_node(state):
    """Use LLM to generate a full answer from the tool output."""
    response = llm.invoke([
        {"role": "system", "content": "You are a helpful assistant. Given the query and tool results, write a clear answer."},
        {"role": "user", "content": f"Query: {state['query']}\nTool output: {state['tool_output']}"}
    ])
    answer = response.content
    print(f"  [GENERATE] {answer[:120]}...")
    return {"answer": answer}

# --- NODE 4: FORMAT ---
def format_node(state):
    """Format the final answer for display."""
    formatted = f"=== FINAL ANSWER ===\n{state['answer']}"
    print(f"  [FORMAT] Done")
    return {"formatted_output": formatted}

# --- BUILD THE GRAPH ---
graph = StateGraph(ResearchState)

graph.add_node("plan", plan_node)
graph.add_node("execute", execute_node)
graph.add_node("generate", generate_node)
graph.add_node("format", format_node)

graph.add_edge("plan", "execute")
graph.add_edge("execute", "generate")
graph.add_edge("generate", "format")
graph.add_edge("format", END)

graph.set_entry_point("plan")
app = graph.compile()

# --- RUN IT ---
print("=" * 60)
print("RUNNING LINEAR GRAPH")
print("=" * 60)
result = app.invoke({
    "query": "What's the weather in Paris?",
    "plan": "",
    "tool_output": "",
    "answer": "",
    "formatted_output": "",
})

print()
print(result["formatted_output"])
print()
print("STATE AFTER GRAPH:")
print(f"  Plan: {result['plan'][:80]}...")
print(f"  Tool Output: {result['tool_output']}")
print(f"  Answer: {result['answer'][:80]}...")

RUNNING LINEAR GRAPH
  [PLAN] 1. Open a reliable weather website or app (e.g., Weather.com, AccuWeather, or a local news site).  
2. Enter “Paris” in ...
  [EXECUTE] Weather in Paris: 18C
  [GENERATE] The current temperature in Paris is about 18 °C....
  [FORMAT] Done

=== FINAL ANSWER ===
The current temperature in Paris is about 18 °C.

STATE AFTER GRAPH:
  Plan: 1. Open a reliable weather website or app (e.g., Weather.com, AccuWeather, or a ...
  Tool Output: Weather in Paris: 18C
  Answer: The current temperature in Paris is about 18 °C....


In [15]:
# =============================================================================
# TASK 3: Add Conditional Edges & Cycles
# =============================================================================
# We add a CRITIQUE node after GENERATE that checks answer quality.
# If score < 7 → route back to GENERATE (retry)
# If score >= 7 → move forward to FORMAT (done)
# max_retries prevents infinite loops
#
# GRAPH:
#   PLAN -> EXECUTE -> GENERATE -> CRITIQUE -> FORMAT -> END
#                              ^           |
#                              +--- RETRY -+
# =============================================================================

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from typing import TypedDict
from langgraph.graph import StateGraph, END

load_dotenv()

# --- LLM ---
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, max_tokens=1024)

# --- TOOLS (from Day 2) ---
@tool
def get_weather(city: str) -> str:
    """Returns current temperature for a given city. Use when user asks about weather."""
    fake_temps = {"tokyo": 22, "paris": 18, "new york": 25, "london": 15}
    temp = fake_temps.get(city.lower(), 20)
    return f"Weather in {city.title()}: {temp}C"

@tool
def product_lookup(product_name: str) -> str:
    """Looks up product info including price and stock. Use when user asks about products."""
    products = {"laptop": 999, "phone": 699, "tablet": 449, "headphones": 149}
    price = products.get(product_name.lower())
    if price:
        return f"Product: {product_name}, Price: ${price}"
    return f"Product '{product_name}' not found"

tools = [get_weather, product_lookup]

# --- STATE SCHEMA ---
# Added: quality_score and retry_count for the self-correction loop
class ResearchState(TypedDict):
    query: str
    plan: str
    tool_output: str
    answer: str
    quality_score: int
    retry_count: int
    max_retries: int
    formatted_output: str

# --- NODE 1: PLAN ---
def plan_node(state):
    response = llm.invoke([
        {"role": "system", "content": "You are a planner. Create a short step-by-step plan. Reply with just the plan."},
        {"role": "user", "content": state["query"]}
    ])
    print(f"  [PLAN] {response.content[:100]}...")
    return {"plan": response.content}

# --- NODE 2: EXECUTE ---
def execute_node(state):
    query = state["query"].lower()
    if "weather" in query or "temperature" in query:
        response = llm.invoke([
            {"role": "system", "content": "Extract the city name. Reply with ONLY the city name."},
            {"role": "user", "content": state["query"]}
        ])
        city = response.content.strip().lower()
        result = get_weather.invoke({"city": city})
    elif "product" in query or "price" in query:
        response = llm.invoke([
            {"role": "system", "content": "Extract the product name. Reply with ONLY the product name."},
            {"role": "user", "content": state["query"]}
        ])
        product = response.content.strip().lower()
        result = product_lookup.invoke({"product_name": product})
    else:
        response = llm.invoke([
            {"role": "system", "content": "Answer briefly in 1-2 sentences."},
            {"role": "user", "content": state["query"]}
        ])
        result = response.content
    print(f"  [EXECUTE] {result[:100]}")
    return {"tool_output": result}

# --- NODE 3: GENERATE ---
# --- NODE 3: GENERATE (improved) ---
def generate_node(state):
    feedback = ""
    if state.get("quality_score", 0) > 0 and state["retry_count"] > 0:
        feedback = f"\nPrevious answer was rated {state['quality_score']}/10. Improve it."
    
    response = llm.invoke([
        {"role": "system", "content": (
            "You are a helpful assistant. Write a clear, complete answer.\n"
            "- For simple questions (what is X, what's the weather): give a direct, confident answer in 1-2 sentences.\n"
            "- For complex questions (explain, compare, how does): give a detailed answer with context.\n"
            "- Never say 'I found' or 'according to results'. Just state the answer directly.\n"
            "- Do not add unnecessary filler or repeat the question."
        )},
        {"role": "user", "content": f"Query: {state['query']}\nTool output: {state['tool_output']}{feedback}"}
    ])
    print(f"  [GENERATE] {response.content[:100]}...")
    return {"answer": response.content}
# --- NODE 4: CRITIQUE (improved) ---
def critique_node(state):
    retry = state["retry_count"]
    response = llm.invoke([
        {"role": "system", "content": (
            "Rate this answer 0-10. Reply with ONLY a number.\n"
            "Scoring rules:\n"
            "- 8-10: Correct, complete, and appropriate length for the question\n"
            "- 5-7: Mostly correct but missing something\n"
            "- 1-4: Wrong, incomplete, or unclear\n"
            "A short answer to a simple question is fine. Do not penalize brevity."
        )},
        {"role": "user", "content": f"Query: {state['query']}\nAnswer: {state['answer']}"}
    ])
    try:
        score = int(response.content.strip())
    except ValueError:
        score = 5
    
    print(f"  [CRITIQUE] Score: {score}/10 (attempt #{retry + 1})")
    return {
        "quality_score": score,
        "retry_count": retry + 1,
    }
# --- CONDITIONAL ROUTER ---
def route_after_critique(state):
    score = state["quality_score"]
    retries = state["retry_count"]
    max_retries = state["max_retries"]
    
    if score >= 7:
        print(f"  [ROUTE] Score {score} >= 7 -> FINISH")
        return "finish"
    elif retries < max_retries:
        print(f"  [ROUTE] Score {score} < 7, retries {retries} < {max_retries} -> RETRY")
        return "retry"
    else:
        print(f"  [ROUTE] Max retries reached -> FINISH")
        return "finish"

# --- NODE 5: FORMAT ---
def format_node(state):
    formatted = f"=== FINAL ANSWER ===\n{state['answer']}"
    print(f"  [FORMAT] Done")
    return {"formatted_output": formatted}

# --- BUILD THE GRAPH ---
graph = StateGraph(ResearchState)

graph.add_node("plan", plan_node)
graph.add_node("execute", execute_node)
graph.add_node("generate", generate_node)
graph.add_node("critique", critique_node)
graph.add_node("format", format_node)

# Linear edges
graph.add_edge("plan", "execute")
graph.add_edge("execute", "generate")
graph.add_edge("generate", "critique")

# Conditional edge: critique decides to retry or finish
graph.add_conditional_edges("critique", route_after_critique, {
    "retry": "generate",   # Bad answer -> go back to generate
    "finish": "format",    # Good answer -> move to format
})

graph.add_edge("format", END)

graph.set_entry_point("plan")
app = graph.compile()

# --- RUN IT ---
print("=" * 60)
print("RUNNING GRAPH WITH SELF-CORRECTION LOOP")
print("=" * 60)
result = app.invoke({
    "query": "What's the weather in Paris?",
    "plan": "", "tool_output": "", "answer": "",
    "quality_score": 0, "retry_count": 0, "max_retries": 3,
    "formatted_output": "",
})

print()
print(result["formatted_output"])
print()
print("FINAL STATE:")
print(f"  Quality Score: {result['quality_score']}/10")
print(f"  Retries: {result['retry_count']}")

RUNNING GRAPH WITH SELF-CORRECTION LOOP
  [PLAN] 1. Open a reliable weather website or app (e.g., Weather.com, AccuWeather, or a local news site).  
...
  [EXECUTE] Weather in Paris: 18C
  [GENERATE] The current temperature in Paris is about 18 °C....
  [CRITIQUE] Score: 6/10 (attempt #1)
  [ROUTE] Score 6 < 7, retries 1 < 3 -> RETRY
  [GENERATE] The current weather in Paris is 18 °C....
  [CRITIQUE] Score: 2/10 (attempt #2)
  [ROUTE] Score 2 < 7, retries 2 < 3 -> RETRY
  [GENERATE] The current temperature in Paris is 18 °C....
  [CRITIQUE] Score: 6/10 (attempt #3)
  [ROUTE] Max retries reached -> FINISH
  [FORMAT] Done

=== FINAL ANSWER ===
The current temperature in Paris is 18 °C.

FINAL STATE:
  Quality Score: 6/10
  Retries: 3


In [21]:
# =============================================================================
# TASK 4: Human-in-the-Loop & Interrupts
# =============================================================================
# Uses interrupt() inside approve node to pause for human approval.
# Uses Command(resume=value) to resume with human input.
#
# GRAPH:
#   PLAN -> EXECUTE -> GENERATE -> CRITIQUE -> APPROVE -> FORMAT -> END
#                              ^           |         |
#                              +--- RETRY -+         +-- REJECT -> END
#
# DISCUSSION: When to use HITL vs Full Autonomy
# ------------------------------------------------
# USE HITL when:
# - Action is irreversible (delete, send, pay)
# - Action has legal/compliance implications
# - Action affects other users
# - Agent confidence is low
#
# FULL AUTONOMY is OK when:
# - Action is reversible (edit, draft)
# - Action only affects the current user
# - Low stakes (recommendation, summary)
# - Agent has high confidence + human can undo
# =============================================================================

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, max_tokens=1024)

@tool
def get_weather(city: str) -> str:
    """Returns current temperature for a given city."""
    fake_temps = {"tokyo": 22, "paris": 18, "new york": 25, "london": 15}
    temp = fake_temps.get(city.lower(), 20)
    return f"Weather in {city.title()}: {temp}C"

@tool
def product_lookup(product_name: str) -> str:
    """Looks up product info including price."""
    products = {"laptop": 999, "phone": 699, "tablet": 449, "headphones": 149}
    price = products.get(product_name.lower())
    if price:
        return f"Product: {product_name}, Price: ${price}"
    return f"Product '{product_name}' not found"

tools = [get_weather, product_lookup]

class ResearchState(TypedDict):
    query: str
    plan: str
    tool_output: str
    answer: str
    quality_score: int
    retry_count: int
    max_retries: int
    formatted_output: str

# --- NODES ---

def plan_node(state):
    response = llm.invoke([
        {"role": "system", "content": "Create a short step-by-step plan. Reply with just the plan."},
        {"role": "user", "content": state["query"]}
    ])
    print(f"  [PLAN] {response.content[:100]}...")
    return {"plan": response.content}

def execute_node(state):
    query = state["query"].lower()
    if "weather" in query or "temperature" in query:
        response = llm.invoke([
            {"role": "system", "content": "Extract the city name. Reply with ONLY the city name."},
            {"role": "user", "content": state["query"]}
        ])
        city = response.content.strip().lower()
        result = get_weather.invoke({"city": city})
    elif "product" in query or "price" in query:
        response = llm.invoke([
            {"role": "system", "content": "Extract the product name. Reply with ONLY the product name."},
            {"role": "user", "content": state["query"]}
        ])
        product = response.content.strip().lower()
        result = product_lookup.invoke({"product_name": product})
    else:
        response = llm.invoke([
            {"role": "system", "content": "Answer briefly in 1-2 sentences."},
            {"role": "user", "content": state["query"]}
        ])
        result = response.content
    print(f"  [EXECUTE] {result[:100]}")
    return {"tool_output": result}

def generate_node(state):
    feedback = ""
    if state.get("quality_score", 0) > 0 and state["retry_count"] > 0:
        feedback = f"\nPrevious answer was rated {state['quality_score']}/10. Improve it."
    response = llm.invoke([
        {"role": "system", "content": (
            "Write a clear, complete answer. "
            "For simple questions: 1-2 sentences. For complex: detailed. "
            "Never say 'I found'. Just state the answer."
        )},
        {"role": "user", "content": f"Query: {state['query']}\nTool output: {state['tool_output']}{feedback}"}
    ])
    print(f"  [GENERATE] {response.content[:100]}...")
    return {"answer": response.content}

def critique_node(state):
    retry = state["retry_count"]
    response = llm.invoke([
        {"role": "system", "content": (
            "Rate this answer 0-10. Reply with ONLY a number.\n"
            "8-10: Correct and complete. 5-7: Mostly correct. 1-4: Wrong.\n"
            "Short answers to simple questions are fine."
        )},
        {"role": "user", "content": f"Query: {state['query']}\nAnswer: {state['answer']}"}
    ])
    try:
        score = int(response.content.strip())
    except ValueError:
        score = 5
    print(f"  [CRITIQUE] Score: {score}/10 (attempt #{retry + 1})")
    return {"quality_score": score, "retry_count": retry + 1}

def route_after_critique(state):
    score = state["quality_score"]
    retries = state["retry_count"]
    max_retries = state["max_retries"]
    if score >= 7:
        return "approve"
    elif retries < max_retries:
        return "retry"
    else:
        return "approve"

# --- APPROVE NODE: Uses interrupt() to pause for human input ---
def approve_node(state):
    """Show draft answer and pause for human approval using interrupt()."""
    print(f"\n  {'='*50}")
    print(f"  HUMAN APPROVAL REQUIRED")
    print(f"  {'='*50}")
    print(f"  Draft Answer: {state['answer']}")
    print(f"  {'='*50}")

    # interrupt() PAUSES the graph here
    # Human must resume with Command(resume="yes") or Command(resume="no")
    approval = interrupt("Approve or reject this answer?")

    print(f"  Human responded: {approval}")
    return {"status": approval}

def route_after_approve(state):
    status = state.get("status", "")
    if status == "yes":
        return "format"
    elif status == "no":
        return "reject"
    else:
        return "format"

def format_node(state):
    formatted = f"=== FINAL ANSWER ===\n{state['answer']}"
    print(f"  [FORMAT] Done")
    return {"formatted_output": formatted}

def reject_node(state):
    print(f"  [REJECT] Answer was rejected. No action taken.")
    return {"formatted_output": "Answer was rejected by human."}

# --- BUILD GRAPH ---
graph = StateGraph(ResearchState)

graph.add_node("plan", plan_node)
graph.add_node("execute", execute_node)
graph.add_node("generate", generate_node)
graph.add_node("critique", critique_node)
graph.add_node("approve", approve_node)
graph.add_node("format", format_node)
graph.add_node("reject", reject_node)

graph.add_edge("plan", "execute")
graph.add_edge("execute", "generate")
graph.add_edge("generate", "critique")

graph.add_conditional_edges("critique", route_after_critique, {
    "retry": "generate",
    "approve": "approve",
})

graph.add_conditional_edges("approve", route_after_approve, {
    "format": "format",
    "reject": "reject",
})

graph.add_edge("format", END)
graph.add_edge("reject", END)

graph.set_entry_point("plan")

memory = MemorySaver()
app = graph.compile(checkpointer=memory)

# ============================================================
# SCENARIO 1: HUMAN APPROVES
# ============================================================
print("=" * 60)
print("SCENARIO 1: Human approves the answer")
print("=" * 60)
config = {"configurable": {"thread_id": "approve-demo"}}

# Run 1: Graph runs to approve node, interrupt() pauses it
print("\n--- Running graph (will pause at approve) ---")
result = app.invoke({
    "query": "What's the weather in Paris?",
    "plan": "", "tool_output": "", "answer": "",
    "quality_score": 0, "retry_count": 0, "max_retries": 3,
    "formatted_output": "",
}, config)
print(f"\nGraph paused. Draft: {result['answer'][:60]}...")

# Run 2: Resume with approval using Command(resume="yes")
print("\n--- Resuming with human approval ---")
result = app.invoke(Command(resume="yes"), config)
print(f"\n{result['formatted_output']}")

# ============================================================
# SCENARIO 2: HUMAN REJECTS
# ============================================================
print("\n" + "=" * 60)
print("SCENARIO 2: Human rejects the answer")
print("=" * 60)
config2 = {"configurable": {"thread_id": "reject-demo"}}

# Run 1: Graph runs to approve node, interrupt() pauses it
print("\n--- Running graph (will pause at approve) ---")
result2 = app.invoke({
    "query": "What's the weather in Paris?",
    "plan": "", "tool_output": "", "answer": "",
    "quality_score": 0, "retry_count": 0, "max_retries": 3,
    "formatted_output": "",
}, config2)
print(f"\nGraph paused. Draft: {result2['answer'][:60]}...")

# Run 2: Resume with rejection using Command(resume="no")
print("\n--- Resuming with human rejection ---")
result2 = app.invoke(Command(resume="no"), config2)
print(f"\nOutput: {result2['formatted_output']}")

# ============================================================
# DISCUSSION: HITL vs Full Autonomy
# ============================================================
print("\n" + "=" * 60)
print("DISCUSSION: When to use HITL vs Full Autonomy")
print("=" * 60)
print("""
USE HITL when:
- Action is irreversible (delete, send, pay)
- Action has legal/compliance implications
- Action affects other users
- Agent confidence is low

FULL AUTONOMY is OK when:
- Action is reversible (edit, draft)
- Action only affects the current user
- Low stakes (recommendation, summary)
- Agent has high confidence + human can undo
""")

SCENARIO 1: Human approves the answer

--- Running graph (will pause at approve) ---
  [PLAN] 1. Choose a reliable weather API (e.g., OpenWeatherMap, WeatherAPI).  
2. Obtain an API key and read...
  [EXECUTE] Weather in Paris: 18C
  [GENERATE] The current weather in Paris is 18 °C....
  [CRITIQUE] Score: 3/10 (attempt #1)
  [GENERATE] The current temperature in Paris is 18 °C (approximately 64 °F)....
  [CRITIQUE] Score: 6/10 (attempt #2)
  [GENERATE] The current temperature in Paris is 18 °C. This indicates mild, comfortable weather....
  [CRITIQUE] Score: 5/10 (attempt #3)

  HUMAN APPROVAL REQUIRED
  Draft Answer: The current temperature in Paris is 18 °C. This indicates mild, comfortable weather.

Graph paused. Draft: The current temperature in Paris is 18 °C. This indicates mi...

--- Resuming with human approval ---

  HUMAN APPROVAL REQUIRED
  Draft Answer: The current temperature in Paris is 18 °C. This indicates mild, comfortable weather.
  Human responded: yes
  [FORMAT] Don

In [22]:
# =============================================================================
# TASK 5: Persistence & Debugging
# =============================================================================
# 1. Checkpointer saves state after every step
# 2. Resume a paused conversation using thread_id
# 3. State history shows what happened at each step
# 4. Time-travel: replay from a previous state
# 5. AgentExecutor vs LangGraph comparison
# =============================================================================

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, max_tokens=1024)

@tool
def get_weather(city: str) -> str:
    """Returns current temperature for a given city."""
    fake_temps = {"tokyo": 22, "paris": 18, "new york": 25, "london": 15}
    temp = fake_temps.get(city.lower(), 20)
    return f"Weather in {city.title()}: {temp}C"

@tool
def product_lookup(product_name: str) -> str:
    """Looks up product info including price."""
    products = {"laptop": 999, "phone": 699, "tablet": 449, "headphones": 149}
    price = products.get(product_name.lower())
    if price:
        return f"Product: {product_name}, Price: ${price}"
    return f"Product '{product_name}' not found"

tools = [get_weather, product_lookup]

class ResearchState(TypedDict):
    query: str
    plan: str
    tool_output: str
    answer: str
    quality_score: int
    retry_count: int
    max_retries: int
    formatted_output: str

# --- NODES ---

def plan_node(state):
    response = llm.invoke([
        {"role": "system", "content": "Create a short step-by-step plan. Reply with just the plan."},
        {"role": "user", "content": state["query"]}
    ])
    print(f"  [PLAN] {response.content[:100]}...")
    return {"plan": response.content}

def execute_node(state):
    query = state["query"].lower()
    if "weather" in query or "temperature" in query:
        response = llm.invoke([
            {"role": "system", "content": "Extract the city name. Reply with ONLY the city name."},
            {"role": "user", "content": state["query"]}
        ])
        city = response.content.strip().lower()
        result = get_weather.invoke({"city": city})
    elif "product" in query or "price" in query:
        response = llm.invoke([
            {"role": "system", "content": "Extract the product name. Reply with ONLY the product name."},
            {"role": "user", "content": state["query"]}
        ])
        product = response.content.strip().lower()
        result = product_lookup.invoke({"product_name": product})
    else:
        response = llm.invoke([
            {"role": "system", "content": "Answer briefly in 1-2 sentences."},
            {"role": "user", "content": state["query"]}
        ])
        result = response.content
    print(f"  [EXECUTE] {result[:100]}")
    return {"tool_output": result}

def generate_node(state):
    feedback = ""
    if state.get("quality_score", 0) > 0 and state["retry_count"] > 0:
        feedback = f"\nPrevious answer was rated {state['quality_score']}/10. Improve it."
    response = llm.invoke([
        {"role": "system", "content": (
            "Write a clear, complete answer. "
            "For simple questions: 1-2 sentences. For complex: detailed. "
            "Never say 'I found'. Just state the answer."
        )},
        {"role": "user", "content": f"Query: {state['query']}\nTool output: {state['tool_output']}{feedback}"}
    ])
    print(f"  [GENERATE] {response.content[:100]}...")
    return {"answer": response.content}

def critique_node(state):
    retry = state["retry_count"]
    response = llm.invoke([
        {"role": "system", "content": (
            "Rate this answer 0-10. Reply with ONLY a number.\n"
            "8-10: Correct and complete. 5-7: Mostly correct. 1-4: Wrong.\n"
            "Short answers to simple questions are fine."
        )},
        {"role": "user", "content": f"Query: {state['query']}\nAnswer: {state['answer']}"}
    ])
    try:
        score = int(response.content.strip())
    except ValueError:
        score = 5
    print(f"  [CRITIQUE] Score: {score}/10 (attempt #{retry + 1})")
    return {"quality_score": score, "retry_count": retry + 1}

def route_after_critique(state):
    score = state["quality_score"]
    retries = state["retry_count"]
    max_retries = state["max_retries"]
    if score >= 7:
        return "approve"
    elif retries < max_retries:
        return "retry"
    else:
        return "approve"

def approve_node(state):
    print(f"\n  {'='*50}")
    print(f"  HUMAN APPROVAL REQUIRED")
    print(f"  {'='*50}")
    print(f"  Draft Answer: {state['answer']}")
    print(f"  {'='*50}")
    approval = interrupt("Approve or reject this answer?")
    print(f"  Human responded: {approval}")
    return {"status": approval}

def route_after_approve(state):
    status = state.get("status", "")
    if status == "yes":
        return "format"
    elif status == "no":
        return "reject"
    else:
        return "format"

def format_node(state):
    formatted = f"=== FINAL ANSWER ===\n{state['answer']}"
    print(f"  [FORMAT] Done")
    return {"formatted_output": formatted}

def reject_node(state):
    print(f"  [REJECT] Answer was rejected. No action taken.")
    return {"formatted_output": "Answer was rejected by human."}

# --- BUILD GRAPH ---
graph = StateGraph(ResearchState)

graph.add_node("plan", plan_node)
graph.add_node("execute", execute_node)
graph.add_node("generate", generate_node)
graph.add_node("critique", critique_node)
graph.add_node("approve", approve_node)
graph.add_node("format", format_node)
graph.add_node("reject", reject_node)

graph.add_edge("plan", "execute")
graph.add_edge("execute", "generate")
graph.add_edge("generate", "critique")

graph.add_conditional_edges("critique", route_after_critique, {
    "retry": "generate",
    "approve": "approve",
})

graph.add_conditional_edges("approve", route_after_approve, {
    "format": "format",
    "reject": "reject",
})

graph.add_edge("format", END)
graph.add_edge("reject", END)

graph.set_entry_point("plan")

# --- COMPILE WITH CHECKPOINTER ---
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

# ============================================================
# PART 1: PERSISTENCE - State persists across runs
# ============================================================
print("=" * 60)
print("PART 1: PERSISTENCE")
print("=" * 60)

config = {"configurable": {"thread_id": "persist-demo"}}

# Run 1: Graph runs and pauses at approve
print("\n--- Run 1: Graph pauses at approve ---")
result1 = app.invoke({
    "query": "What's the weather in Paris?",
    "plan": "", "tool_output": "", "answer": "",
    "quality_score": 0, "retry_count": 0, "max_retries": 3,
    "formatted_output": "",
}, config)
print(f"Paused. Draft: {result1['answer'][:50]}...")

# Run 2: Resume later (simulated) - state was saved by checkpointer
print("\n--- Run 2: Resume later (state was saved) ---")
result2 = app.invoke(Command(resume="yes"), config)
print(f"Completed: {result2['formatted_output'][:50]}...")

# ============================================================
# PART 2: STATE HISTORY - See what happened at each step
# ============================================================
print("\n" + "=" * 60)
print("PART 2: STATE HISTORY")
print("=" * 60)

# Get full history for this thread
print("\n--- State history for persist-demo ---")
history = list(app.get_state_history(config))

for i, state_snapshot in enumerate(history):
    node_values = state_snapshot.values
    print(f"\n  Step {i}:")
    print(f"    Query: {node_values.get('query', 'N/A')[:40]}...")
    print(f"    Plan: {node_values.get('plan', 'N/A')[:40]}...")
    print(f"    Answer: {node_values.get('answer', 'N/A')[:40]}...")
    print(f"    Quality: {node_values.get('quality_score', 'N/A')}")
    print(f"    Retries: {node_values.get('retry_count', 'N/A')}")

# ============================================================
# PART 3: TIME-TRAVEL - Replay from a previous state
# ============================================================
print("\n" + "=" * 60)
print("PART 3: TIME-TRAVEL (Replay)")
print("=" * 60)

# Pick a state from history to replay from
if len(history) > 1:
    target_state = history[1]  # Replay from second step
    print(f"\n--- Replaying from step 1 ---")
    print(f"  State at step 1: {target_state.values.get('plan', 'N/A')[:60]}...")
    
    # Create new thread for replay
    replay_config = {"configurable": {"thread_id": "replay-demo"}}
    
    # Replay by invoking with the target state's config
    # This starts a new run but from the saved state
    replay_result = app.invoke(
        target_state.values,
        {"configurable": {"thread_id": "replay-demo"}}
    )
    print(f"  Replay result: {replay_result.get('answer', 'N/A')[:60]}...")

# ============================================================
# PART 4: AGENTEXECUTOR vs LANGGRAPH COMPARISON
# ============================================================
print("\n" + "=" * 60)
print("PART 4: AGENTEXECUTOR vs LANGGRAPH")
print("=" * 60)
print("""
| Feature              | AgentExecutor              | LangGraph                    |
|----------------------|----------------------------|------------------------------|
| Architecture         | Flat loop                  | Graph (nodes + edges)        |
| Control Flow         | Automatic (LLM decides)    | Explicit (you define)        |
| Conditional Logic    | Hard to add                | Native                       |
| Cycles/Loops         | Can't loop back            | Can loop to any node         |
| HITL                 | Not supported              | interrupt() + Command()      |
| Persistence          | Manual                     | Checkpointer built-in        |
| Debugging            | Black box                  | State history + time-travel  |
| State                | Implicit (messages)        | Explicit (TypedDict)         |

USE AGENTEXECUTOR WHEN:
- Simple agent: LLM picks tool, runs it, repeats
- No branching logic needed
- No human approval needed
- Quick prototype

USE LANGGRAPH WHEN:
- Multiple paths through workflow
- Self-correction loops (retry if bad answer)
- Human-in-the-loop required
- Need persistence and debugging
- Complex multi-step processes
""")

PART 1: PERSISTENCE

--- Run 1: Graph pauses at approve ---
  [PLAN] 1. Choose a reliable weather API (e.g., OpenWeatherMap, WeatherAPI).  
2. Obtain an API key and read...
  [EXECUTE] Weather in Paris: 18C
  [GENERATE] The current weather in Paris is 18 °C....
  [CRITIQUE] Score: 4/10 (attempt #1)
  [GENERATE] The current temperature in Paris is 18 °C....
  [CRITIQUE] Score: 2/10 (attempt #2)
  [GENERATE] The current temperature in Paris is 18 °C....
  [CRITIQUE] Score: 2/10 (attempt #3)

  HUMAN APPROVAL REQUIRED
  Draft Answer: The current temperature in Paris is 18 °C.
Paused. Draft: The current temperature in Paris is 18 °C....

--- Run 2: Resume later (state was saved) ---

  HUMAN APPROVAL REQUIRED
  Draft Answer: The current temperature in Paris is 18 °C.
  Human responded: yes
  [FORMAT] Done
Completed: === FINAL ANSWER ===
The current temperature in Pa...

PART 2: STATE HISTORY

--- State history for persist-demo ---

  Step 0:
    Query: What's the weather in Paris?...
    P